In [0]:
silver_df = spark.read.table("mashebhebs_nyctaxi.silver_nyctaxi")

In [0]:
from pyspark.sql import functions as F

silver_df = silver_df.withColumn("pickup_date", F.date_format("tpep_pickup_datetime", "yyyy-MM-dd"))

In [0]:
gold_daily_revenue_df = silver_df.groupBy("pickup_date").agg(F.sum("fare_amount").alias("total_day_fare"))


In [0]:
peak_hours_df = silver_df \
    .withColumn("pickup_hour", F.hour(F.col("tpep_pickup_datetime"))) \
    .withColumn(
        "duration_minutes",
        F.expr("cast(split(trip_duration, 'h')[0] as int) * 60 + cast(split(trip_duration, 'h')[1] as int)")
    )

peak_hours_df = peak_hours_df \
    .groupBy("pickup_hour") \
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg(F.col("duration_minutes")), 2).alias("avg_trip_duration_mins")
    ) \
    .orderBy("pickup_hour")

display(peak_hours_df.limit(10))

In [0]:
gold_df = peak_hours_df.write \
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("mashebhebs_nyctaxi.gold_nyctaxi")